In [4]:
import weaviate
from weaviate.classes.config import Configure, DataType, Multi2VecField, Property
import os
import json


In [5]:
client = weaviate.connect_to_local(
    host="172.17.0.2",  
    port=8080,
    grpc_port=50051,
 
)
print(client.is_ready())

collections = client.collections.delete("unified_embedding")


True


In [6]:
questions = client.collections.create(
    "unified_embedding",
    properties=[
        Property(name="type", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="page", data_type=DataType.INT, skip_vectorization=True),
        
        # Fields for embedding (truncated)
        Property(name="description_preview", data_type=DataType.TEXT, skip_vectorization=False),
        Property(name="text_preview", data_type=DataType.TEXT, skip_vectorization=False),
        
        # Fields for storage (full text)
        Property(name="description", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="text", data_type=DataType.TEXT, skip_vectorization=True),
        
        Property(name="trace", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="filename", data_type=DataType.TEXT, skip_vectorization=True),
        Property(name="image", data_type=DataType.BLOB, skip_vectorization=False),
    ],
    vectorizer_config=Configure.Vectorizer.multi2vec_clip(
        image_fields=[
            Multi2VecField(name="image", weight=0.5)
        ],
        text_fields=[
            Multi2VecField(name="description_preview", weight=0.25),
            Multi2VecField(name="text_preview", weight=0.25)
        ]
    )
)

/home/prime/miniconda3/envs/embedder/lib/python3.11/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [7]:
def smart_truncate(text, max_tokens=70):
    """Truncate text intelligently to fit CLIP's 77 token limit"""
    if not text or len(text) == 0:
        return ""
    
    import re
    
    # Check if text is primarily decorative characters
    clean_text = text.replace('\\', '')
    if re.match(r'^[_\-\s]{20,}$', clean_text):
        return ""
    
    # Replace long sequences of escaped underscores/dashes
    text = re.sub(r'(\\_){10,}', '___', text)
    text = re.sub(r'(_){10,}', '___', text)
    text = re.sub(r'(-){10,}', '___', text)
    
    # Remove special characters that inflate token counts
    text = re.sub(r'\\[\[\]\(\)]', '', text)  # Remove escaped brackets: \[ \] \( \)
    text = re.sub(r'[\[\]\(\)]', '', text)     # Remove regular brackets: [ ] ( )
    text = re.sub(r'\\', '', text)              # Remove remaining backslashes
    
    # Clean up multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # CLIP tokenizer roughly: 1 token ≈ 4 characters (more conservative now)
    max_chars = max_tokens * 3.5  # ~245 characters for safety
    
    if len(text) <= max_chars:
        return text
    
    # Truncate
    truncated = text[:int(max_chars)]
    
    # Try to end at a sentence
    last_period = truncated.rfind('.')
    if last_period > max_chars * 0.7:
        return truncated[:last_period + 1]
    
    return truncated

In [5]:
# Get the collection
collection = client.collections.get("unified_embedding")

for filename in sorted(os.listdir("../clean_chunks")):
    if filename.endswith(".json"):
        filepath = os.path.join("../clean_chunks", filename)
        
        with open(filepath, 'r') as f:
            data = json.load(f)
        
        print(f"Importing data from {filename} with {len(data)} entries...")
        
        with collection.batch.fixed_size(batch_size=200) as batch:
            for d in data:
                properties = {
                    "type": d["block_type"],
                    "page": d["page"],
                    
                    # Full text for storage/retrieval
                    "description": d.get("description", ""),
                    "text": d.get("text", ""),
                    
                    # Truncated text for embedding
                    "description_preview": smart_truncate(d.get("description", ""), 35),
                    "text_preview": smart_truncate(d.get("text", ""), 35),
                    
                    "trace": d["trace"],
                    "filename": d["filename"],
                }

                # Handle image
                if d.get("images"):
                    if isinstance(d["images"], dict):
                        properties["image"] = list(d["images"].values())[0]
                    else:
                        properties["image"] = d["images"]
                else:
                    properties["image"] = None
            
                batch.add_object(properties=properties)
                
                if batch.number_errors > 10:
                    print("Batch import stopped due to excessive errors.")
                    break

        # Check for failed objects
        failed_objects = collection.batch.failed_objects
        if failed_objects:
            print(f"Number of failed imports: {len(failed_objects)}")
            print(f"Failed on filename: {filepath}")
            print(f"First failed object: {failed_objects[0]}")
        
        print(f"Finished importing data from {filename}")

client.close()

Importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json with 627 entries...
Finished importing data from O-RAN-WG1-CCIN-TR-R004-v01.00_cleaned.json
Importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json with 384 entries...
Finished importing data from O-RAN-WG6.AppLCM-Deployment-R003-v02.00_cleaned.json
Importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json with 413 entries...
Finished importing data from O-RAN.SFG.Non-RT-RIC-Security-TR-v01.00_cleaned.json
Importing data from O-RAN.SuFG.CE-v01.00_cleaned.json with 181 entries...
Finished importing data from O-RAN.SuFG.CE-v01.00_cleaned.json
Importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json with 339 entries...
Finished importing data from O-RAN.SuFG.TR.NES-Analysis-R004-v01.01_cleaned.json
Importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json with 279 entries...
Finished importing data from O-RAN.TIFG.CGofOTIC.0-v06.00_cleaned.json
Importing data from O-RAN.TIFG.E

{'message': 'Failed to send 1 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 78 and max_position_embeddings: 77'}}
{'message': 'Failed to send 1 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send 1 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 107 and max_position_embeddings: 77'}}
{'message': 'Failed to send 1 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 2
Failed on filename: ../clean_chunks/O-RAN.TIFG.TS.E2E-Test.0-R004-v08.00_cleaned.json
First failed object: ErrorObject(message='fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 78 and max_position_embeddings: 77', object_=BatchObject(collection='Unified_embedding', properties={'type': 'Table', 'page': 52, 'description': '', 'text': '| Channel bandwidth BW [MHz]  \n---|---  \nμ| | 10| 15| 20| 25| 30| 40| 50| 60| 80| 90| 100| 200| 400  \n0| 25| 52| 79| 106| 133| 160| 216| 270| N/A| N/A| N/A| N/A| N/A| N/A  \n| 11| 24| 38| 51| 65| 78| 106| 133| 162| 217| 245| 273| N/A| N/A  \n2| N/A| 11| 18| 24| 31| 38| 51| 65| 79| 107| 121| 135| N/A| N/A  \n3| N/A| N/A| N/A| N/A| N/A| N/A| N/A| 66| N/A| N/A| N/A| 132| 264| N/A  \n4| N/A| N/A| N/A| N/A| N/A| N/A| N/A| 32| N/A| N/A| N/A| 66| 132| 264\n\n', 'description_preview': '', 'text_preview': '| Channel bandwidth BW MHz ---|--- μ| | 10| 15| 20| 25| 30| 40| 50| 60| 80| 

{'message': 'Failed to send 1 in a batch of 105', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 80 and max_position_embeddings: 77'}}
{'message': 'Failed to send 1 objects in a batch of 105. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 1
Failed on filename: ../clean_chunks/O-RAN.WG1.TS.Slicing-Architecture-R004-v14.01_cleaned.json
First failed object: ErrorObject(message='fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 80 and max_position_embeddings: 77', object_=BatchObject(collection='Unified_embedding', properties={'type': 'Table', 'page': 77, 'description': '', 'text': '2022.03.26| 06.00.09| Addition of CR:  \n---|---|---  \n| | -  \nRBBN.AO-2022.03.02-WG1-CR-0001-Annex-TN-Slicing-v02  \n2022.03.26| 06.00.10| Changes accepted from v06.00.09  \n| | Baseline for WG1 approval  \n2022.04.04| 07.00| Final version 07.00  \n2022.07.24| 07.00.01| Initial version towards v08.00, starting with v07.00.01\nper O-RAN specification revision  \nnumbering process  \n2022.07.25| 07.00.02| Addition of CR:  \n| | -  \nNC.AO-2022.05.02-WG1-NC-0004-MultiOperatorRANSliceSubnetAnnexure-v02  \n| | Editorial updates and corrections  \n2022.07.25| 07.00.03| 

...

Number of failed imports: 14
Failed on filename: ../clean_chunks/O-RAN.WG9.XTRP-TST.0-R004-v05.00 (1)_cleaned.json
First failed object: ErrorObject(message='fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 96 and max_position_embeddings: 77', object_=BatchObject(collection='Unified_embedding', properties={'type': 'Table', 'page': 131, 'description': '', 'text': 'Packet max|TEL  \n|| ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠|  \n---|---|---|---  \nPacket max|TERL|| ≤ 60𝑛𝑠(𝐹𝑅2)| ≤ 100𝑛𝑠(𝐹𝑅1)|  \n| ≤ 190𝑛𝑠(𝐹𝑅1)| |   \n1PPS max|TEL  \n|| ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠|  \nFrequency Limit  \n(For O-DU Class  \nA)| ≤ 15𝑝𝑝𝑏| ≤ 15𝑝𝑝𝑏|  \nFrequency Limit  \n(For O-DU Class  \nB)| ≤ 5𝑝𝑝𝑏| ≤ 5𝑝𝑝𝑏|  \nRelative Time Error between the O-RU UNI ports connected to O-DU1 and O-DU2|  \nPacket max|TERL|| ≤ 2930𝑛𝑠| ≤ 2840𝑛𝑠|  \n1PPS max|TERL|| ≤ 2930𝑛𝑠| ≤ 2840𝑛𝑠|\n\n', 'description_preview': '', 'text_preview': 'Packet max|TEL || ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠| ---|---|---|--- Packet max|TERL|| ≤ 

{'message': 'Failed to send 2 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 90 and max_position_embeddings: 77', 'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 96 and max_position_embeddings: 77'}}
{'message': 'Failed to send 2 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send 1 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 80 and max_position_embeddings: 77'}}
{'message': 'Failed to send 1 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send 4 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_positi

Number of failed imports: 14
Failed on filename: ../clean_chunks/O-RAN.WG9.XTRP-TST.0-R004-v05.00_cleaned.json
First failed object: ErrorObject(message='fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 96 and max_position_embeddings: 77', object_=BatchObject(collection='Unified_embedding', properties={'type': 'Table', 'page': 131, 'description': '', 'text': 'Packet max|TEL  \n|| ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠|  \n---|---|---|---  \nPacket max|TERL|| ≤ 60𝑛𝑠(𝐹𝑅2)| ≤ 100𝑛𝑠(𝐹𝑅1)|  \n| ≤ 190𝑛𝑠(𝐹𝑅1)| |   \n1PPS max|TEL  \n|| ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠|  \nFrequency Limit  \n(For O-DU Class  \nA)| ≤ 15𝑝𝑝𝑏| ≤ 15𝑝𝑝𝑏|  \nFrequency Limit  \n(For O-DU Class  \nB)| ≤ 5𝑝𝑝𝑏| ≤ 5𝑝𝑝𝑏|  \nRelative Time Error between the O-RU UNI ports connected to O-DU1 and O-DU2|  \nPacket max|TERL|| ≤ 2930𝑛𝑠| ≤ 2840𝑛𝑠|  \n1PPS max|TERL|| ≤ 2930𝑛𝑠| ≤ 2840𝑛𝑠|\n\n', 'description_preview': '', 'text_preview': 'Packet max|TEL || ≤ 1465𝑛𝑠| ≤ 1420𝑛𝑠| ---|---|---|--- Packet max|TERL|| ≤ 60𝑛𝑠

{'message': 'Failed to send 1 in a batch of 200', 'errors': {'fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 104 and max_position_embeddings: 77'}}
{'message': 'Failed to send 1 objects in a batch of 200. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


Number of failed imports: 1
Failed on filename: ../clean_chunks/ORAN-WG3.E2SM-NI-v01.00_cleaned.json
First failed object: ErrorObject(message='fail with status 500: Sequence length must be less than max_position_embeddings (got `sequence length`: 104 and max_position_embeddings: 77', object_=BatchObject(collection='Unified_embedding', properties={'type': 'Text', 'page': 33, 'description': '', 'text': '18<br>19<br>20<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>23<br>24<br>25<br>26<br>27<br>28<br>29<br>30<br>31<br>32<br>33<br>34<br>35<br>36<br>37<br>38', 'description_preview': '', 'text_preview': '18<br>19<br>20<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>22<br>23<br>24<br>25<br>26<br>27<br>28<br>29<br>30<br>31', 'trace': '8.4 Information Element Abstract Syntax \\(with ASN.1\\) --> 8.4.1 General 2', 'filename': 'ORAN-WG3.E2SM-NI-v01.00', 'image': None}, references=None, uuid='15b22b1f-2b07-4b1c-b3c3-c0b3e539e58c', vector=None, tenant=None, index=541, retry_count=0), 

In [10]:
from PIL import Image
import io
import base64

client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
)

collection = client.collections.get("unified_embedding")

response = collection.query.near_text(
    query="5G RAN architecture diagram",
    limit=10,
    return_properties=["type", "page", "description", "text", "filename", "image"]
)

print(f"Found {len(response.objects)} results\n")

for obj in response.objects:
    print(f"{'='*80}")
    print(f"Type: {obj.properties['type']}")
    print(f"Page: {obj.properties['page']}")
    print(f"Filename: {obj.properties['filename']}")
    print(f"Text Preview: {obj.properties['text'] if obj.properties['text'] else 'N/A'}")
    print(f"\nDescription: {obj.properties['description'] if obj.properties['description'] else 'N/A'}...")
    
    # Display image if it exists
    if obj.properties.get("image"):
        try:
            image_b64 = obj.properties["image"]
            image_bytes = base64.b64decode(image_b64)
            image = Image.open(io.BytesIO(image_bytes))
            image.show()
            print("[Image displayed]\n")
        except Exception as e:
            print(f"[Could not display image: {e}]\n")
    else:
        print("[No image]\n")
client.close()

Found 10 results

Type: Text
Page: 14
Filename: O-RAN.WG9.XTRP-REQ-v01.00
Text Preview: As outlined 5G defines different services and RAN architectures. Figure 2 is adapted from NGMN \[56\] and illustrates how different RAN components may be placed in different locations within the

Description: N/A...
[No image]

Type: SectionHeader
Page: 36
Filename: O-RAN.WG9.XPSAAS.0-R004-v09.00
Text Preview: Figure 7-4: 5G Backhaul components and interfaces Source: Adapted from 3GPP TS 23.501  v6.4.0\(2020-03\): System Architecture for 5G \[1\] with control plane / user plane shading added  by document authors.

Description: N/A...
[No image]

Type: SectionHeader
Page: 28
Filename: O-RAN.WG9.XTRP-REQ-v01.00
Text Preview: Figure 10: 5G Backhaul components and interfaces Source: Adapted from 3GPP TS 23.501 v6.4.0\(2020-03\): System Architecture for 5G \[67\] with  control plane / user plane shading added by document authors.

Description: N/A...
[No image]

Type: Text
Page: 80
Filename: O-RAN.WG1.TS